In [18]:
# for working part
import numpy as np
import pandas as pd

# for ml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler,PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

In [2]:
revenue_prediction = pd.read_parquet('E:\\Python New\\revenue-intelligence\\data\\processed_data_files\\revenue_prediction_dataset.parquet')

customer_segmentation = pd.read_parquet('E:\\Python New\\revenue-intelligence\\data\\processed_data_files\\customer_segmentation_dataset.parquet')

repeat_prediction = pd.read_parquet('E:\\Python New\\revenue-intelligence\\data\\processed_data_files\\repeat_purchase_prediction_dataset.parquet')

product_recomendation = pd.read_parquet('E:\\Python New\\revenue-intelligence\\data\\processed_data_files\\product_recomendation_dataset.parquet')

In [7]:
x1 = revenue_prediction[['lead_source', 'profile', 'channel', 'state', 'category','order_count',
                         'unique_customers', 'qty','rolling_revenue_7D', 'moving_avg_7D',
                           'previous_revenue_1D','previous_revenue_7D', 'previous_revenue_30D',
                             'aov', 'order_month','order_year', 'day_of_week', 'is_weekend']]

y1 = revenue_prediction[['sales_amt']]


x2 = customer_segmentation[['clv', 'order_count', 'unique_sku_purchased',
                             'unique_category_count','total_qty', 'aov', 'delivered_rate',
                             'rto_rate','average_basket_size', 'average_monthly_order_frequency',
                             'average_days_between_orders']]


x3 = repeat_prediction[['clv', 'order_count', 'aov', 'average_basket_size',
                        'average_monthly_order_frequency', 'average_days_between_orders']]

y3 = repeat_prediction[['repeat_purchase']]

In [8]:
categorical_x1 = ['lead_source', 'profile', 'channel', 'state', 'category','order_month','order_year','day_of_week']

numerical_x1 = ['order_count','unique_customers', 'qty','rolling_revenue_7D', 'moving_avg_7D','previous_revenue_1D',
                'previous_revenue_7D', 'previous_revenue_30D','aov']

binary_x1 = ['is_weekend']


numerical_x2 = ['clv', 'order_count', 'unique_sku_purchased','unique_category_count','total_qty', 'aov', 'delivered_rate',
                'rto_rate','average_basket_size', 'average_monthly_order_frequency','average_days_between_orders']


numerical_x3 = ['clv', 'order_count', 'aov', 'average_basket_size','average_monthly_order_frequency', 'average_days_between_orders']


In [10]:
x1_train,x1_test,y1_train,y1_test = train_test_split(x1,y1,random_state=42,test_size=0.2)
x3_train,x3_test,y3_train,y3_test = train_test_split(x3,y3,random_state=42,stratify=y3,test_size=0.2)


In [ ]:
dataset_config = {'revenue_prediction': {"problem_type":"regression",
                                         "x_train":x1_train,
                                         "x_test":x1_test,
                                         "y_train":y1_train,
                                         "y_test":y1_test,
                                         
                                         'cat':categorical_x1,
                                         'num':numerical_x1,
                                         'binary':binary_x1},


                    'repeat_prediction':{"problem_type":"classification",
                                         "x_train":x3_train,
                                         "x_test":x3_test,
                                         "y_train":y3_train,
                                         "y_test":y3_test,
                                         
                                         'cat': None,
                                         'num':numerical_x3,
                                         'binary': None},
                                         
                    'customer_segmentation':{"problem_type":"clustering",
                                             'x':x2,
                                             
                                             'cat':None,
                                             'num':numerical_x2,
                                             'binary':None}}

In [16]:
def scale_pos_weight(y_train):
    return y_train.value_counts()[0] / y_train.value_counts()[1]

In [15]:
def preprocess(categorical_cols=None,numerical_cols=None, binary_cols = None,transformation=None):
    transformers = []

    if categorical_cols:
        transformers.append('cat',OneHotEncoder(handle_unknown='ignore'),categorical_cols)

    if numerical_cols:
        if transformation == None:
            num_pipeline = Pipeline([('scaler',StandardScaler())])
        elif transformation == 'power':
            num_pipeline = Pipeline([('power',PowerTransformer(method='yeo-johnson')),
                                     ('scaler',StandardScaler())])
            
        transformers.append('num',num_pipeline,numerical_cols)

    if binary_cols:
        transformers.append('binary','passthrough',binary_cols)

    preprocessor = ColumnTransformer(transformers=transformers)

    return preprocessor
            

In [ ]:
def create_models(problem_type):
    if problem_type == 'regression':
        models = {'LR':LinearRegression(),
                  'XGB':XGBRegressor()}

    elif problem_type == 'classification':
        models = {}

    elif problem_type == 'clustering':
        models = {}

    return models